In [0]:
from pyspark.sql import functions as F, Window
CATALOG, SCHEMA = "workspace", "olist"

def silver(n): return spark.table(f"{CATALOG}.{SCHEMA}.silver_{n}")

def save_gold(df, n):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_{n}"))
    print("saved gold_" + n, spark.table(f"{CATALOG}.{SCHEMA}.gold_{n}").count(), "rows")

delivered = silver("sales").filter("order_status = 'delivered'")

In [0]:
# MONTHLY REVENUE + MoM GROWTH  (Window + lag() = SQL's LAG() OVER (ORDER BY month))
def monthly(df):
    m = (df.withColumn("month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
           .groupBy("month")
           .agg(F.countDistinct("order_id").alias("orders"), F.sum("price").alias("revenue")))
    w = Window.orderBy("month")
    return (m.withColumn("prev_month_revenue", F.lag("revenue").over(w))
             .withColumn("mom_growth_pct",
                         F.when(F.col("prev_month_revenue") > 0,
                                F.round(100 * (F.col("revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue"), 1)))
             .withColumn("revenue", F.round("revenue", 2))
             .withColumn("prev_month_revenue", F.round("prev_month_revenue", 2))
             .orderBy("month"))

save_gold(monthly(delivered), "monthly_revenue")
save_gold(monthly(delivered.filter("date_format(order_purchase_timestamp,'yyyy-MM') BETWEEN '2017-01' AND '2018-08'")),
          "monthly_revenue_dashboard")                                  # your 02b version: Jan 2017 - Aug 2018

# TOP CATEGORIES with share and cumulative share  (rank() and sum() over windows)
tot = Window.partitionBy()                                              # one window over the whole table
by_rev = Window.orderBy(F.desc("revenue"))                              # running total from biggest to smallest
cat = (delivered.groupBy("category")
       .agg(F.countDistinct("order_id").alias("orders"), F.sum("price").alias("revenue"))
       .withColumn("revenue_rank", F.rank().over(by_rev))
       .withColumn("pct_of_revenue", F.round(100 * F.col("revenue") / F.sum("revenue").over(tot), 2))
       .withColumn("cumulative_pct", F.round(100 * F.sum("revenue").over(by_rev) / F.sum("revenue").over(tot), 2))
       .withColumn("revenue", F.round("revenue", 2))
       .orderBy("revenue_rank"))
save_gold(cat, "top_categories")

# ORDERS AND REVENUE BY STATE
st = (delivered.groupBy("customer_state")
      .agg(F.countDistinct("order_id").alias("orders"), F.sum("price").alias("revenue"))
      .withColumnRenamed("customer_state", "state")
      .withColumn("revenue_rank", F.rank().over(by_rev))
      .withColumn("pct_of_orders", F.round(100 * F.col("orders") / F.sum("orders").over(tot), 2))
      .withColumn("pct_of_revenue", F.round(100 * F.col("revenue") / F.sum("revenue").over(tot), 2))
      .withColumn("cumulative_pct", F.round(100 * F.sum("revenue").over(by_rev) / F.sum("revenue").over(tot), 2))
      .withColumn("revenue", F.round("revenue", 2))
      .orderBy("revenue_rank"))
save_gold(st, "state_sales")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


saved gold_monthly_revenue 23 rows
saved gold_monthly_revenue_dashboard 20 rows
saved gold_top_categories 74 rows
saved gold_state_sales 27 rows


In [0]:
# DELIVERY: one row per delivered order that has a delivery date (days = difference in seconds / 86400)
deliv = (silver("orders")
         .filter("order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL")
         .withColumn("days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400)
         .withColumn("est_days", (F.unix_timestamp("order_estimated_delivery_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400)
         .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)))

by_state = (deliv.join(silver("customers").select("customer_id", "customer_state"), "customer_id", "left")
            .groupBy(F.col("customer_state").alias("state"))
            .agg(F.count("*").alias("orders"),
                 F.round(F.avg("days"), 1).alias("avg_delivery_days"),
                 F.round(100 * F.sum("is_late") / F.count("*"), 1).alias("pct_late"))
            .orderBy(F.desc("avg_delivery_days")))
save_gold(by_state, "delivery_by_state")

# KPIs: one row for the dashboard counters
k_sales = delivered.agg(F.countDistinct("order_id").alias("delivered_orders"),
                        F.count("*").alias("items_sold"),
                        F.round(F.sum("price"), 2).alias("total_revenue"),
                        F.round(F.sum("freight_value"), 2).alias("total_freight"))
k_aov = (delivered.groupBy("order_id").agg(F.sum("price").alias("v"))
         .agg(F.round(F.avg("v"), 2).alias("aov")))
k_del = deliv.agg(F.count("*").alias("delivered_with_date"),
                  F.round(F.avg("days"), 1).alias("avg_delivery_days"),
                  F.round(F.avg("est_days"), 1).alias("avg_estimated_days"),
                  F.round(100 * F.sum("is_late") / F.count("*"), 1).alias("pct_late"))
save_gold(k_sales.crossJoin(k_aov).crossJoin(k_del), "kpis")

display(spark.table(f"{CATALOG}.{SCHEMA}.gold_kpis"))

saved gold_delivery_by_state 27 rows
saved gold_kpis 1 rows


delivered_orders,items_sold,total_revenue,total_freight,aov,delivered_with_date,avg_delivery_days,avg_estimated_days,pct_late
96478,110197,13221498.11,2198275.64,137.04,96470,12.6,23.7,8.1
